## HW 2 - Exercise (a)

For a 3D Cartesian grid with `N` points per dimension, the basis size is: $M = N^3$.

The dense Hamiltonian has $M^2 = N^6$ floating-point entries.

If each float uses 8 bytes, the RAM required is approximately: $\text{RAM}(N) \approx 8N^6$ bytes.

## HW 2 - Exercise (b)

For a 3D Cartesian grid, $\hat{H}$ has dimension $M\times M$ with $M=N^3$.
Assuming 7 nonzeros per row:
$$\text{nnz} \approx 7M = 7N^3$$

CSR stores three arrays:
- `data`: `nnz` floats $\Rightarrow 7N^3\,b_f$ bytes
- `indices`: `nnz` integers $\Rightarrow 7N^3\,b_i$ bytes
- `indptr`: `M+1=N^3+1` integers $\Rightarrow (N^3+1)\,b_i$ bytes

So the total RAM is:
$$\text{RAM}_{\text{CSR}} \approx 7N^3\,b_f + 7N^3\,b_i + (N^3+1)\,b_i = 7N^3\,b_f + (8N^3+1)\,b_i\ \text{bytes}.$$

If `data=float64` and integer arrays are `int32`:
$$\text{RAM}_{\text{CSR}} \approx 7N^3\cdot 8 + (8N^3+1)\cdot 4 = 88N^3 + 4\ \text{bytes}.$$

## HW 2 - Exercise (c)

The following code plots memory (GB) for dense and sparse storage over $N\in[10,200]$, and finds the smallest $N$ where each exceeds 16 GB.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sweep grid size N to compare dense-vs-CSR memory scaling.
N = np.arange(10, 201)

# Storage assumptions used in the analytic model.
b_float = 8  # bytes for float64
b_int = 4    # bytes for int32
laptop_bytes = 16 * (1024**3)  # 16 GiB machine limit

# Dense model from part (a): 8*N^6 bytes.
dense_bytes = 8 * N**6

# CSR model from part (b): 7*N^3 floats + (8*N^3+1) ints.
sparse_bytes = 7 * N**3 * b_float + (8 * N**3 + 1) * b_int  # = 88*N^3 + 4

# Convert bytes to decimal GB for plotting.
to_gb = 1e9
dense_gb = dense_bytes / to_gb
sparse_gb = sparse_bytes / to_gb

# Critical N where each model exceeds 16 GiB.
N_dense_crit = int(np.ceil((laptop_bytes / 8) ** (1/6)))
N_sparse_crit = int(np.ceil(((laptop_bytes - 4) / 88) ** (1/3)))

print(f'Dense exceeds 16 GiB at N >= {N_dense_crit}')
print(f'Sparse (CSR) exceeds 16 GiB at N >= {N_sparse_crit}')

# Plot both curves and mark threshold crossings.
plt.figure(figsize=(8, 5))
plt.plot(N, dense_gb, label='Dense H (8N^6 bytes)', linewidth=2)
plt.plot(N, sparse_gb, label='Sparse CSR H (88N^3+4 bytes)', linewidth=2)
plt.axhline(laptop_bytes / to_gb, color='k', linestyle='--', label='16 GiB laptop RAM')
plt.axvline(N_dense_crit, color='tab:blue', linestyle=':', alpha=0.8)
if 10 <= N_sparse_crit <= 200:
    plt.axvline(N_sparse_crit, color='tab:orange', linestyle=':', alpha=0.8)

plt.xlabel('N')
plt.ylabel('Memory (GB)')
plt.title('Dense vs Sparse Memory Requirement')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## HW 2 - Exercise (d)

Build the dense 3D kinetic operator using Kronecker products of 1D $\hat{T}$ and identity matrices for several $N\leq 13$. Then compare measured memory from `T.nbytes` to the algebraic scaling from (a), $8N^6$ bytes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Atomic-unit constants used throughout the assignment.
hbar = 1.0
m = 1.0

def T1D_dense(N):
    """
    Return dense 1D kinetic operator using the 2nd-order finite-difference Laplacian.
    """
    main = -2.0 * np.ones(N)
    off = 1.0 * np.ones(N - 1)
    lap1d = np.diag(main) + np.diag(off, 1) + np.diag(off, -1)
    return -(hbar**2 / (2.0 * m)) * lap1d

# Use small N so dense 3D matrices still fit in memory.
N_vals = np.arange(2, 14)  # N <= 13
actual_bytes = []
model_bytes = []

for N in N_vals:
    # Build 3D kinetic operator with Kronecker sum.
    T = T1D_dense(N)
    I = np.eye(N)
    T3 = np.kron(np.kron(T, I), I) + np.kron(np.kron(I, T), I) + np.kron(np.kron(I, I), T)

    # Measured dense memory and model prediction from part (a).
    actual_bytes.append(T3.nbytes)
    model_bytes.append(8 * N**6)

actual_bytes = np.array(actual_bytes)
model_bytes = np.array(model_bytes)

print('N, actual bytes, model bytes (8N^6):')
for N, a, b in zip(N_vals, actual_bytes, model_bytes):
    print(f'{N:2d}: {a:12d}  {b:12d}')

# Visual check that measured and model curves agree.
plt.figure(figsize=(8, 5))
plt.plot(N_vals, actual_bytes / 1e9, 'o-', label='Measured (T3.nbytes)')
plt.plot(N_vals, model_bytes / 1e9, 's--', label='Model from (a): 8N^6 bytes')
plt.xlabel('N')
plt.ylabel('Memory (GB)')
plt.title('Dense 3D $\hat{T}$ Memory: Measured vs Algebraic')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## HW 2 - Exercise (e)

Build the 3D kinetic operator $\hat{T}$ as a CSR sparse matrix using `scipy.sparse`, then compare measured CSR memory to the algebraic model from (b).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse

# Atomic-unit constants.
hbar = 1.0
m = 1.0

def T1D_sparse(N):
    """Return 1D kinetic operator in CSR format (3-point stencil)."""
    pref = -(hbar**2) / (2.0 * m)
    main = -2.0 * np.ones(N)
    off = np.ones(N - 1)
    return sparse.diags([off, main, off], offsets=[-1, 0, 1], format='csr') * pref

def T3D_sparse(N):
    """Build 3D kinetic operator as Kronecker sum: T?I?I + I?T?I + I?I?T."""
    T = T1D_sparse(N)
    I = sparse.eye(N, format='csr')
    T3 = (
        sparse.kron(sparse.kron(T, I, format='csr'), I, format='csr')
        + sparse.kron(sparse.kron(I, T, format='csr'), I, format='csr')
        + sparse.kron(sparse.kron(I, I, format='csr'), T, format='csr')
    )
    return T3.tocsr()

# Test multiple N values and compare measured CSR memory to analytic part (b) model.
N_vals = np.arange(4, 51)
measured_bytes = []
model_b_bytes = []
nnz_vals = []

for N in N_vals:
    T3 = T3D_sparse(N)

    # True CSR memory = data + indices + indptr arrays.
    measured = T3.data.nbytes + T3.indices.nbytes + T3.indptr.nbytes
    measured_bytes.append(measured)
    nnz_vals.append(T3.nnz)

    # Part (b) model: 7N^3*b_f + (8N^3+1)*b_i.
    b_f = T3.data.itemsize
    b_i = T3.indices.itemsize
    model_b = 7 * (N**3) * b_f + (8 * (N**3) + 1) * b_i
    model_b_bytes.append(model_b)

measured_bytes = np.array(measured_bytes, dtype=np.int64)
model_b_bytes = np.array(model_b_bytes, dtype=np.int64)
nnz_vals = np.array(nnz_vals, dtype=np.int64)

print('N, nnz(T3), measured bytes, model-(b) bytes:')
for N, nnz, meas, mod in zip(N_vals[::6], nnz_vals[::6], measured_bytes[::6], model_b_bytes[::6]):
    print(f'{N:2d}: nnz={nnz:10d}, measured={meas:12d}, model(b)={mod:12d}')

plt.figure(figsize=(8, 5))
plt.plot(N_vals, measured_bytes / 1e6, 'o-', label='Measured CSR memory')
plt.plot(N_vals, model_b_bytes / 1e6, 's--', label='Model from (b)')
plt.xlabel('N')
plt.ylabel('Memory (MB)')
plt.title('3D Sparse $\hat{T}$ Memory: Measured vs Algebraic Model')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

rel_err = (measured_bytes - model_b_bytes) / model_b_bytes
print(f'Min/Max relative error over N range: {rel_err.min():.4f} to {rel_err.max():.4f}')


## HW 2 - Exercise (f)`n
`n
Interpret the 3D sparse $\\hat{T}$ as the full Hamiltonian for a 3D particle in a box with Dirichlet boundary conditions, and verify convergence of the lowest 7 eigenvalues to analytic values.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import eigsh

# Particle-in-a-box constants in atomic units.
hbar = 1.0
m = 1.0
L = 1.0

def T1D_dirichlet_interior(N_total, L=1.0, hbar=1.0, m=1.0):
    """
    1D kinetic operator on interior points only for 0<=x<=L with Dirichlet BCs.
    N_total includes both boundary nodes; matrix dimension is N_total-2.
    """
    n = N_total - 2
    dx = L / (N_total - 1)
    pref = -(hbar**2) / (2.0 * m * dx**2)
    main = -2.0 * np.ones(n)
    off = np.ones(n - 1)
    return sparse.diags([off, main, off], offsets=[-1, 0, 1], format='csr') * pref

def H3D_dirichlet(N_total, L=1.0, hbar=1.0, m=1.0):
    """3D Hamiltonian H=T for a free particle in a box (Dirichlet interior grid)."""
    T = T1D_dirichlet_interior(N_total, L=L, hbar=hbar, m=m)
    n = N_total - 2
    I = sparse.eye(n, format='csr')
    H = (
        sparse.kron(sparse.kron(T, I, format='csr'), I, format='csr')
        + sparse.kron(sparse.kron(I, T, format='csr'), I, format='csr')
        + sparse.kron(sparse.kron(I, I, format='csr'), T, format='csr')
    )
    return H.tocsr()

def analytic_lowest7(L=1.0, hbar=1.0, m=1.0):
    """Analytic lowest 7 energies with degeneracies: [3,6,6,6,9,9,9] * pi^2/2."""
    vals = np.array([3, 6, 6, 6, 9, 9, 9], dtype=float)
    pref = (np.pi**2 * hbar**2) / (2.0 * m * L**2)
    return pref * vals

N_list = [21, 31, 41, 51]
E_ref = analytic_lowest7(L=L, hbar=hbar, m=m)

abs_err_max = []
rel_err_max = []

print('Reference lowest 7 energies:')
print(np.array2string(E_ref, precision=8))
print('\nN_total, dim, max|E_num-E_ref|, max relative error')

for N in N_list:
    # Build sparse Hamiltonian and solve smallest 7 eigenvalues.
    H = H3D_dirichlet(N, L=L, hbar=hbar, m=m)
    dim = H.shape[0]
    evals, _ = eigsh(H, k=7, which='SA', tol=1e-10)
    evals = np.sort(evals)

    # Store error metrics for convergence plot.
    abs_err = np.abs(evals - E_ref)
    rel_err = abs_err / E_ref
    abs_err_max.append(abs_err.max())
    rel_err_max.append(rel_err.max())

    print(f'{N:2d}, {dim:7d}, {abs_err.max():.6e}, {rel_err.max():.6e}')
    print('  E_num =', np.array2string(evals, precision=8))
    print('  E_ref =', np.array2string(E_ref, precision=8))

abs_err_max = np.array(abs_err_max)
rel_err_max = np.array(rel_err_max)

plt.figure(figsize=(8, 5))
plt.plot(N_list, abs_err_max, 'o-', label='Max absolute error (lowest 7)')
plt.plot(N_list, rel_err_max, 's--', label='Max relative error (lowest 7)')
plt.xlabel('N_total (grid points per axis, including boundaries)')
plt.ylabel('Error')
plt.title('Convergence of Lowest 7 3D Box Eigenvalues (Dirichlet)')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Exercise 2(a) - 3D Coulomb Potential Diagonal

Construct the diagonal entries for $\hat{V}$ in the same basis ordering used for the Kronecker-built 3D operators.

In [ ]:
import numpy as np
from scipy import sparse

def coulomb_diagonal_3d(N, epsilon=1e-2, L=1.0, x_min=None, x_max=None):
    """
    Build softened Coulomb potential diagonal V(r) = -1/sqrt(r^2 + epsilon^2)
    on an N x N x N Cartesian grid in atomic units.

    Basis ordering used everywhere in this notebook:
        p = i*N*N + j*N + k    (k index is fastest).
    """
    if N < 1:
        raise ValueError('N must be >= 1')
    if epsilon < 0:
        raise ValueError('epsilon must be >= 0')

    # Default to symmetric box if explicit bounds are not provided.
    if x_min is None or x_max is None:
        x_min = -0.5 * L
        x_max = 0.5 * L

    # Same 1D grid for x, y, z.
    xyz = np.linspace(x_min, x_max, N)

    # indexing='ij' gives X[i,j,k],Y[i,j,k],Z[i,j,k].
    # ravel(order='C') then matches p = i*N*N + j*N + k.
    X, Y, Z = np.meshgrid(xyz, xyz, xyz, indexing='ij')
    r2 = X**2 + Y**2 + Z**2
    v = -1.0 / np.sqrt(r2 + epsilon**2)

    # Flatten to diagonal ordering used by sparse.diags.
    v_diag = v.ravel(order='C')
    return v_diag, xyz

def coulomb_matrix_3d_csr(N, epsilon=1e-2, L=1.0, x_min=None, x_max=None):
    """Create sparse diagonal matrix V from coulomb_diagonal_3d."""
    v_diag, xyz = coulomb_diagonal_3d(N, epsilon=epsilon, L=L, x_min=x_min, x_max=x_max)
    V = sparse.diags(v_diag, offsets=0, format='csr')
    return V, v_diag, xyz

# Quick sanity check for dimensions and values.
N_test = 5
V_test, vdiag_test, grid_test = coulomb_matrix_3d_csr(N_test, epsilon=0.2, L=2.0)
print('N=', N_test, 'diag length=', vdiag_test.shape[0], 'matrix shape=', V_test.shape)
print('center grid value (x=0 if included):', grid_test[N_test // 2])
print('min/max(Vdiag)=', vdiag_test.min(), vdiag_test.max())


## Exercise 2(b) - Build $\hat{V}$ and Compute Ground-State Energy

Use `scipy.sparse.diags()` with the diagonal from part (a), then build $\hat{H}=\hat{T}+\hat{V}$ on $-7.5 \le x,y,z \le 7.5$ and compute the ground-state energy for requested grid sizes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import eigsh

# Atomic units.
hbar = 1.0
m = 1.0

def coulomb_diagonal_interior(N_total, x_min=-7.5, x_max=7.5, epsilon=1e-2):
    """
    Coulomb diagonal on interior points only (Dirichlet boundaries excluded).
    Flattened index map is p = i*n*n + j*n + k, where n = N_total - 2.
    """
    if N_total < 3:
        raise ValueError('N_total must be >= 3 for Dirichlet interior points.')
    if epsilon < 0:
        raise ValueError('epsilon must be >= 0.')

    # Full grid includes boundaries; interior grid excludes endpoints.
    x_full = np.linspace(x_min, x_max, N_total)
    x = x_full[1:-1]

    # Build V(x,y,z) and flatten to match Kronecker ordering.
    X, Y, Z = np.meshgrid(x, x, x, indexing='ij')
    r2 = X**2 + Y**2 + Z**2
    v_diag = (-1.0 / np.sqrt(r2 + epsilon**2)).ravel(order='C')
    return v_diag, x

def T1D_dirichlet_interior(N_total, x_min=-7.5, x_max=7.5, hbar=1.0, m=1.0):
    """1D kinetic matrix on interior points for domain [x_min, x_max]."""
    n = N_total - 2
    dx = (x_max - x_min) / (N_total - 1)
    pref = -(hbar**2) / (2.0 * m * dx**2)
    main = -2.0 * np.ones(n)
    off = np.ones(n - 1)
    return sparse.diags([off, main, off], offsets=[-1, 0, 1], format='csr') * pref

def H_hydrogen_box_sparse(N_total, x_min=-7.5, x_max=7.5, epsilon=1e-2, hbar=1.0, m=1.0):
    """Assemble full hydrogen Hamiltonian H = T + V in sparse CSR form."""
    # --- Kinetic part (Kronecker sum) ---
    T = T1D_dirichlet_interior(N_total, x_min=x_min, x_max=x_max, hbar=hbar, m=m)
    n = N_total - 2
    I = sparse.eye(n, format='csr')
    T3 = (
        sparse.kron(sparse.kron(T, I, format='csr'), I, format='csr')
        + sparse.kron(sparse.kron(I, T, format='csr'), I, format='csr')
        + sparse.kron(sparse.kron(I, I, format='csr'), T, format='csr')
    ).tocsr()

    # --- Potential part (required sparse.diags) ---
    v_diag, x_int = coulomb_diagonal_interior(N_total, x_min=x_min, x_max=x_max, epsilon=epsilon)
    V3 = sparse.diags(v_diag, offsets=0, format='csr')

    # Final Hamiltonian.
    H = (T3 + V3).tocsr()
    return H, x_int

# N values requested in the prompt (duplicate 26 kept intentionally).
N_list = [20, 26, 30, 26, 40]
x_min, x_max = -7.5, 7.5
epsilon = 1e-2

E0_vals = []

print('N_total, dim, E0 (Hartree), |E0 - (-0.5)|')
for N in N_list:
    # Solve for the smallest algebraic eigenvalue = ground-state energy.
    H, _ = H_hydrogen_box_sparse(N, x_min=x_min, x_max=x_max, epsilon=epsilon, hbar=hbar, m=m)
    e0, _ = eigsh(H, k=1, which='SA', tol=1e-9)
    E0 = float(e0[0])
    E0_vals.append(E0)
    print(f'{N:2d}, {H.shape[0]:7d}, {E0:+.8f}, {abs(E0 + 0.5):.8e}')

E0_vals = np.array(E0_vals)

# Scatter comparison against analytic hydrogen value E0 = -0.5 Ha.
plt.figure(figsize=(8, 5))
plt.scatter(N_list, E0_vals, label='Numerical E0')
plt.scatter(N_list, [-0.5] * len(N_list), label='Analytic E0 = -0.5 Hartree')
plt.xlabel('N_total (points/axis including boundaries)')
plt.ylabel('Ground-state energy (Hartree)')
plt.title('Hydrogen in a 3D Box: Ground-State Convergence')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Exercise 2(c) - Extract $\psi(x,0,0)$ for $N=40$

For even grids there is no exact $y=z=0$ point, so use the two nearest interior grid points to zero for $y$ and $z$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.linalg import eigsh

# Use N=40 as requested.
N = 40
x_min, x_max = -7.5, 7.5
epsilon = 1e-2

# Build Hamiltonian from part (b).
H, x_int = H_hydrogen_box_sparse(N, x_min=x_min, x_max=x_max, epsilon=epsilon, hbar=1.0, m=1.0)

# Ground-state eigenpair.
evals, evecs = eigsh(H, k=1, which='SA', tol=1e-9)
E0 = float(evals[0])
psi0 = evecs[:, 0]

# Reshape flat eigenvector back to 3D grid with the same index convention.
n = N - 2
psi0_3d = psi0.reshape((n, n, n), order='C')  # p = i*n*n + j*n + k

# For even N, x=0 is not an interior grid node. Pick nearest +/- points to 0.
order = np.argsort(np.abs(x_int))
i_neg, i_pos = sorted([int(order[0]), int(order[1])], key=lambda idx: x_int[idx])
y_neg, y_pos = x_int[i_neg], x_int[i_pos]

# Extract four nearest (y,z) combinations around (0,0).
rho_nn = np.abs(psi0_3d[:, i_neg, i_neg])**2  # (-,-)
rho_pp = np.abs(psi0_3d[:, i_pos, i_pos])**2  # (+,+)
rho_np = np.abs(psi0_3d[:, i_neg, i_pos])**2  # (-,+)
rho_pn = np.abs(psi0_3d[:, i_pos, i_neg])**2  # (+,-)

# Average is a better proxy for |psi(x,0,0)|^2 on even grids.
rho_avg4 = 0.25 * (rho_nn + rho_pp + rho_np + rho_pn)

print(f'E0 (N=40) = {E0:+.8f} Hartree')
print(f'Nearest interior coords to 0: y,z in {{{y_neg:+.8f}, {y_pos:+.8f}}}')

# Plot each proxy curve plus their average.
plt.figure(figsize=(8, 5))
plt.scatter(x_int, rho_nn, s=14, label=f'|psi(x,{y_neg:+.3f},{y_neg:+.3f})|^2')
plt.scatter(x_int, rho_pp, s=14, label=f'|psi(x,{y_pos:+.3f},{y_pos:+.3f})|^2')
plt.scatter(x_int, rho_np, s=14, label=f'|psi(x,{y_neg:+.3f},{y_pos:+.3f})|^2')
plt.scatter(x_int, rho_pn, s=14, label=f'|psi(x,{y_pos:+.3f},{y_neg:+.3f})|^2')
plt.scatter(x_int, rho_avg4, s=10, label='Average of 4 nearest slices')
plt.xlabel('x (a.u.)')
plt.ylabel('Density |psi|^2')
plt.title('Ground-State Density Slice Near y=z=0 (N=40)')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Exercise 2(d) - Compare to Analytic 1s Density Along x

Compare the numerical near-$(y,z)=(0,0)$ slice from part (c) to
$$P(x)=\frac{\Delta x}{\pi a_0^3}e^{-2|x|/a_0},\quad a_0=1.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Use exactly the x-grid from part (c).
# rho_avg4 is the numerical near-(y,z)=(0,0) proxy from part (c).
dx = x_int[1] - x_int[0]
a0 = 1.0

# Analytic 1s line probability on the same x samples.
P_analytic = (dx / (np.pi * a0**3)) * np.exp(-2.0 * np.abs(x_int) / a0)

# Overlay numerical and analytic curves.
plt.figure(figsize=(8, 5))
plt.scatter(x_int, rho_avg4, s=14, label='Numerical: avg of 4 nearest (y,z) slices')
plt.scatter(x_int, P_analytic, s=14, label='Analytic 1s line probability P(x)')
plt.xlabel('x (a.u.)')
plt.ylabel('Density / line probability')
plt.title('Numerical Near-(0,0) Slice vs Analytic 1s Along x')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Print summary values to compare absolute scale.
print(f'dx = {dx:.8f}')
print(f'Max numerical (rho_avg4) = {rho_avg4.max():.8e}')
print(f'Max analytic  P(x)      = {P_analytic.max():.8e}')


### Interpretation
1. **Magnitude difference**: the numerical curve in (c) is a pointwise 3D density sample (using nearby $y,z$ points), while $P(x)$ includes a factor of $\Delta x$ and is a **line probability mass per x-bin**. They are related but not identical observables, so absolute scales differ.
2. **Peak near 0**: for the 1s state, $|\psi(r)|^2 \propto e^{-2r/a_0}$ is largest at $r=0$. Along the x-axis with $y=z=0$, this gives a maximum at $x=0$.
3. **Peak shape**: the analytic profile has a cusp in slope at $x=0$ because it depends on $|x|$. The numerical curve is broadened/rounded by finite grid spacing, softening parameter $\epsilon$, and using nearest off-axis points instead of exact $y=z=0$.

## Exercise 2(e) - First Five States at N=40 and Degeneracy Check

Compute the low-lying spectrum for $N=40$, compare to analytic hydrogen energies $E_n=-1/(2n^2)$, and test whether the $n=2$ degeneracy is broken.

In [ ]:
import numpy as np
from scipy.sparse.linalg import eigsh

# Same Hamiltonian setup as part (b)/(c).
N = 40
x_min, x_max = -7.5, 7.5
epsilon = 1e-2

H, x_int = H_hydrogen_box_sparse(N, x_min=x_min, x_max=x_max, epsilon=epsilon, hbar=1.0, m=1.0)

# Request a few extra states so the clustered n=2 manifold is robustly captured.
evals, evecs = eigsh(H, k=8, which='SA', tol=1e-11, ncv=80)
e = np.sort(evals)

# Energy ordering for first five physical states in this run:
# 1s, then three 2p-like states, then 2s-like state.
E_1s = e[0]
E_2p = e[1:4]
E_2s = e[4]

# Analytic hydrogen energies in atomic units: E_n = -1/(2 n^2).
E_ref_1 = -0.5
E_ref_2 = -0.125

# Relative-error metrics.
rel_1s = abs(E_1s - E_ref_1) / abs(E_ref_1)
rel_2p = np.abs(E_2p - E_ref_2) / abs(E_ref_2)
rel_2s = abs(E_2s - E_ref_2) / abs(E_ref_2)

print('Lowest 8 eigenvalues (Hartree):')
print(np.array2string(e, precision=10))
print()
print('State            E_num            E_ref        Relative error')
print(f'1s      {E_1s:+.10f}   {E_ref_1:+.10f}   {rel_1s:.6e}')
for i, Ei in enumerate(E_2p, start=1):
    print(f'2p[{i}]   {Ei:+.10f}   {E_ref_2:+.10f}   {rel_2p[i-1]:.6e}')
print(f'2s      {E_2s:+.10f}   {E_ref_2:+.10f}   {rel_2s:.6e}')

# Degeneracy diagnostics.
split_2p = E_2p.max() - E_2p.min()
split_n2_total = max(E_2s, E_2p.max()) - min(E_2s, E_2p.min())
print()
print(f'2p splitting (max-min): {split_2p:.6e} Hartree')
print(f'Total n=2 manifold spread (including 2s): {split_n2_total:.6e} Hartree')

if split_2p > 1e-5:
    print('Conclusion: 2p triplet degeneracy is numerically broken.')
else:
    print('Conclusion: 2p triplet remains degenerate to numerical precision.')

if abs(E_2s - E_2p.mean()) > 1e-3:
    print('Conclusion: 2s is split from 2p (n=2 accidental degeneracy is broken).')
else:
    print('Conclusion: 2s and 2p remain nearly degenerate.')


## Exercise 2(f) - Visualize the Four n=2 States in the xz-Plane

Use `plt.contourf()` for the four n=2 states from part (e).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.linalg import eigsh

# Same N=40 Hamiltonian.
N = 40
x_min, x_max = -7.5, 7.5
epsilon = 1e-2

H, x_int = H_hydrogen_box_sparse(N, x_min=x_min, x_max=x_max, epsilon=epsilon, hbar=1.0, m=1.0)

# Solve and sort low-lying eigenpairs.
evals, evecs = eigsh(H, k=8, which='SA', tol=1e-11, ncv=80)
idx = np.argsort(evals)
evals = evals[idx]
evecs = evecs[:, idx]

n = N - 2
# n=2 manifold from part (e): first four excited states.
n2_energies = evals[1:5]
n2_vecs = evecs[:, 1:5]

# Even N has no y=0 interior node; choose nearest positive y to zero.
i_near = np.where(x_int > 0)[0][0]
y_slice = x_int[i_near]

# Coordinates for contour plotting in xz plane.
X, Z = np.meshgrid(x_int, x_int, indexing='ij')

fig, axes = plt.subplots(2, 2, figsize=(11, 9), constrained_layout=True)
axes = axes.ravel()

for s in range(4):
    # Convert flat eigenvector to 3D grid.
    psi3 = n2_vecs[:, s].reshape((n, n, n), order='C')

    # Take xz slice at fixed y ~ 0.
    psi_xz = psi3[:, i_near, :]

    # Symmetric color scale around zero emphasizes nodal structure.
    vmax = np.max(np.abs(psi_xz))
    levels = np.linspace(-vmax, vmax, 41)
    cs = axes[s].contourf(X, Z, psi_xz, levels=levels, cmap='seismic')
    fig.colorbar(cs, ax=axes[s], shrink=0.85)
    axes[s].set_xlabel('x (a.u.)')
    axes[s].set_ylabel('z (a.u.)')
    axes[s].set_title(f'n=2 state {s + 1}, E={n2_energies[s]:+.8f}')

plt.suptitle(f'n=2 manifold in xz plane at y={y_slice:+.6f} (nearest to 0)', y=1.02)
plt.show()

# Report splitting diagnostics used in the discussion.
print('n=2 energies (Hartree):', np.array2string(n2_energies, precision=10))
print('2p-like triplet splitting (first 3 in n=2 manifold):', float(n2_energies[:3].max() - n2_energies[:3].min()))
print('2s-vs-2p centroid split:', float(n2_energies[3] - np.mean(n2_energies[:3])))


### Interpretation
1. **Which states remain degenerate?** The three 2p-like states remain (numerically) degenerate to solver precision, while the 2s state is shifted relative to the 2p triplet, so full n=2 accidental degeneracy is broken.
2. **Why this happens for N=40**: with an even grid, there is no interior point exactly at the origin or exactly at y=0 slice. The Cartesian discretization and finite box/domain break perfect rotational symmetry, so states that should be degenerate in the continuum can split.
3. **Why p-state nodal planes look this way**: p orbitals are odd along one Cartesian direction and have a nodal plane through the origin. On this grid, the nodal plane can look slightly shifted/broadened because the plotted plane is at y ~ dx/2 rather than y=0 and because finite-difference discretization mixes nearby degenerate basis functions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

H, _ = H_hydrogen_box_sparse(41, -7.5, 7.5, 1e-5, 1, 1)

ground_energy, ground_state = eigsh(H, k=1)

print(ground_energy)

size=(39,39,39)

ground_state = ground_state.reshape(size)

print(ground_state[19,19,19])


[-99978.66742521]
0.9999999962072076


Since the softening parameter approaches zero, this means that the potential energy blows up at 0,0,0. In fact, the potential energy at this point is of the order -10^3. At the nearby points, the potential energy is many magnitudes of order lower, specifically sqrt(3)*(15/41). This is over three magnitudes of order lower, and depending on the exact values may even be closer to three and a half orders lower.

The expectation value of the kinetic energy at this point is roughly the same as at any other nearby point, and indeed globally, with the exception of boundary conditions. Therefore the potential energy dominates. 

This implies that the lowest energy of the system will likely be found at this dominant basis state. The expectation value of the wavefunction at this point, will be at the very least 3.5 orders of magnitude higher than at other points, and this is indeed what we observe in our calculations(In fact, we observe nearly eight orders of magnitude higher, but since it is normalised to one there is not much difference between three and a half and eight orders). 
